# 📊 Student Productivity & Distraction — Colab Notebook

This notebook mirrors the **same dataset and models** used in your web app:

- Loads `student_productivity_distraction_dataset_20000.csv`
- Trains:
  - A **Random Forest regressor** to predict `productivity_score`
  - A **Random Forest classifier** for 3-class `stress_band` (low / medium / high)
- Evaluates model performance
- Runs example predictions for a new student
- Shows a few quick visual insights (correlation map + group plots)

> **Before you start in Colab:**
> - Upload `student_productivity_distraction_dataset_20000.csv` to the Colab working directory (or mount Google Drive and update `DATA_PATH`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

DATA_PATH = "student_productivity_distraction_dataset_20000.csv"  # update if stored elsewhere


In [ ]:
# 1. Load dataset and inspect basic structure
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 1.1 Basic statistics

This is similar to the **Statistics** step in the frontend pipeline.
It shows summary statistics (count, mean, quartiles) for all numeric columns.

In [ ]:
# Summary statistics for numeric columns (like the stats cards in the UI)
df.describe().T

## 2. Feature sets for the two models

These match the backend logic:
- **Productivity model** uses study / sleep / phone / distraction features + stress_level.
- **Stress model** predicts a 3-class band (low / medium / high) from the same habits plus `productivity_score`.


In [ ]:
PRODUCTIVITY_FEATURES = [
    "age",
    "study_hours_per_day",
    "sleep_hours",
    "phone_usage_hours",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours",
    "breaks_per_day",
    "coffee_intake_mg",
    "exercise_minutes",
    "assignments_completed",
    "attendance_percentage",
    "stress_level",
]

STRESS_FEATURES = [
    "age",
    "study_hours_per_day",
    "sleep_hours",
    "phone_usage_hours",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours",
    "breaks_per_day",
    "coffee_intake_mg",
    "exercise_minutes",
    "assignments_completed",
    "attendance_percentage",
    "productivity_score",
]


In [ ]:
def train_models(df: pd.DataFrame, random_state: int = 42):
    """Train the two models used in the app.

    - RandomForestRegressor for productivity_score
    - RandomForestClassifier for 3-class stress_band (low/medium/high)
    """
    # ---- Productivity model (regression) ----
    X_prod = df[PRODUCTIVITY_FEATURES]
    y_prod = df["productivity_score"]

    X_train_prod, X_test_prod, y_train_prod, y_test_prod = train_test_split(
        X_prod, y_prod, test_size=0.2, random_state=random_state
    )

    prod_model = RandomForestRegressor(
        n_estimators=200, random_state=random_state, n_jobs=-1
    )
    prod_model.fit(X_train_prod, y_train_prod)

    y_pred_prod = prod_model.predict(X_test_prod)
    prod_mse = mean_squared_error(y_test_prod, y_pred_prod)
    prod_rmse = float(np.sqrt(prod_mse))
    prod_r2 = float(r2_score(y_test_prod, y_pred_prod))

    # ---- Stress model (3-class classification) ----
    if "stress_level" not in df.columns:
        raise ValueError("Column 'stress_level' not found in dataset")

    bins = [0, 3, 7, 10]
    labels = ["low", "medium", "high"]
    df_local = df.copy()
    df_local["stress_band"] = pd.cut(
        df_local["stress_level"], bins=bins, labels=labels, include_lowest=True
    )

    X_stress = df_local[STRESS_FEATURES]
    y_stress = df_local["stress_band"]

    X_train_stress, X_test_stress, y_train_stress, y_test_stress = train_test_split(
        X_stress,
        y_stress,
        test_size=0.2,
        random_state=random_state,
        stratify=y_stress,
    )

    stress_model = RandomForestClassifier(
        n_estimators=200, random_state=random_state, n_jobs=-1
    )
    stress_model.fit(X_train_stress, y_train_stress)

    y_pred_stress = stress_model.predict(X_test_stress)
    stress_accuracy = float(accuracy_score(y_test_stress, y_pred_stress))

    return {
        "prod_model": prod_model,
        "stress_model": stress_model,
        "X_test_prod": X_test_prod,
        "y_test_prod": y_test_prod,
        "X_test_stress": X_test_stress,
        "y_test_stress": y_test_stress,
        "metrics": {
            "productivity": {
                "mse": float(prod_mse),
                "rmse": prod_rmse,
                "r2_score": prod_r2,
            },
            "stress": {
                "accuracy": stress_accuracy,
            },
        },
    }


In [ ]:
# 3. Train models and inspect metrics
bundle = train_models(df)
bundle["metrics"]


In [ ]:
def predict_productivity(models, **kwargs) -> float:
    """Single-student productivity prediction (same interface as backend)."""
    model = models["prod_model"]
    data = pd.DataFrame(
        {
            "age": [kwargs["age"]],
            "study_hours_per_day": [kwargs["study_hours_per_day"]],
            "sleep_hours": [kwargs["sleep_hours"]],
            "phone_usage_hours": [kwargs["phone_usage_hours"]],
            "social_media_hours": [kwargs["social_media_hours"]],
            "youtube_hours": [kwargs["youtube_hours"]],
            "gaming_hours": [kwargs["gaming_hours"]],
            "breaks_per_day": [kwargs["breaks_per_day"]],
            "coffee_intake_mg": [kwargs["coffee_intake_mg"]],
            "exercise_minutes": [kwargs["exercise_minutes"]],
            "assignments_completed": [kwargs["assignments_completed"]],
            "attendance_percentage": [kwargs["attendance_percentage"]],
            "stress_level": [kwargs["stress_level"]],
        }
    )
    X = data[PRODUCTIVITY_FEATURES]
    pred = model.predict(X)[0]
    return float(round(pred, 2))


def predict_stress_band(models, **kwargs) -> str:
    """Single-student stress-band prediction (low / medium / high)."""
    model = models["stress_model"]
    data = pd.DataFrame(
        {
            "age": [kwargs["age"]],
            "study_hours_per_day": [kwargs["study_hours_per_day"]],
            "sleep_hours": [kwargs["sleep_hours"]],
            "phone_usage_hours": [kwargs["phone_usage_hours"]],
            "social_media_hours": [kwargs["social_media_hours"]],
            "youtube_hours": [kwargs["youtube_hours"]],
            "gaming_hours": [kwargs["gaming_hours"]],
            "breaks_per_day": [kwargs["breaks_per_day"]],
            "coffee_intake_mg": [kwargs["coffee_intake_mg"]],
            "exercise_minutes": [kwargs["exercise_minutes"]],
            "assignments_completed": [kwargs["assignments_completed"]],
            "attendance_percentage": [kwargs["attendance_percentage"]],
            "productivity_score": [kwargs["productivity_score"]],
        }
    )
    X = data[STRESS_FEATURES]
    pred = model.predict(X)[0]
    return str(pred)


In [ ]:
# 4. Example end-to-end prediction for a single student
example = dict(
    age=22,
    study_hours_per_day=4.0,
    sleep_hours=7.0,
    phone_usage_hours=5.0,
    social_media_hours=2.0,
    youtube_hours=1.0,
    gaming_hours=1.0,
    breaks_per_day=5,
    coffee_intake_mg=200.0,
    exercise_minutes=30.0,
    assignments_completed=5,
    attendance_percentage=80.0,
    stress_level=5,
)

prod_pred = predict_productivity(bundle, **example)
print("Predicted productivity score:", prod_pred)

stress_input = example.copy()
stress_input.pop("stress_level")
stress_input["productivity_score"] = prod_pred
stress_band = predict_stress_band(bundle, **stress_input)
print("Predicted stress band:", stress_band)


## 5.1 Gender quick insights (coffee & exercise)

This cell recreates the **By Gender** quick insight card from the frontend:
- Average coffee intake (mg) by gender
- Average exercise minutes by gender

In [ ]:
sns.set_style("whitegrid")

if "gender" not in df.columns:
    raise ValueError("Column 'gender' not found in dataset")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

coffee = df.groupby("gender")["coffee_intake_mg"].mean().sort_values()
axes[0].barh(coffee.index, coffee.values, color="saddlebrown")
axes[0].set_xlabel("Average Coffee Intake (mg)")
axes[0].set_title("Coffee Intake by Gender")
axes[0].grid(True, alpha=0.3, axis="x")

exercise = df.groupby("gender")["exercise_minutes"].mean().sort_values()
axes[1].barh(exercise.index, exercise.values, color="mediumseagreen")
axes[1].set_xlabel("Average Exercise Minutes")
axes[1].set_title("Exercise by Gender")
axes[1].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## 5.2 Stress-band quick insights (productivity & coffee)

This cell recreates the **By Stress Band** quick insight card from the frontend:
- Average productivity score for low / medium / high stress bands
- Average coffee intake (mg) for each stress band

In [ ]:
sns.set_style("whitegrid")

if "stress_level" not in df.columns:
    raise ValueError("Column 'stress_level' not found in dataset")

bins = [0, 3, 7, 10]
labels = ["low", "medium", "high"]
df_band = df.copy()
df_band["stress_band"] = pd.cut(df_band["stress_level"], bins=bins, labels=labels, include_lowest=True)

bands = df_band.groupby("stress_band")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

prod = bands["productivity_score"].mean()
axes[0].bar(prod.index.astype(str), prod.values, color=["#34A853", "#FFB300", "#EA4335"])
axes[0].set_ylabel("Average Productivity Score")
axes[0].set_title("Productivity vs Stress Band")
axes[0].grid(True, alpha=0.3, axis="y")

coffee = bands["coffee_intake_mg"].mean()
axes[1].bar(coffee.index.astype(str), coffee.values, color=["#34A853", "#FFB300", "#EA4335"])
axes[1].set_ylabel("Average Coffee Intake (mg)")
axes[1].set_title("Coffee Intake vs Stress Band")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 5.3 Custom distribution explorer

Like the **Distribution explorer** in the frontend, set `DIST_COLUMN` below
and re-run the cell to see a histogram + boxplot for that feature.

In [ ]:
# Change this to any numeric column name (e.g., 'productivity_score', 'stress_level', 'study_hours_per_day')
DIST_COLUMN = "productivity_score"

if DIST_COLUMN not in df.columns:
    raise ValueError(f"Column {DIST_COLUMN!r} not found in dataset")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[DIST_COLUMN], bins=40, edgecolor="black", alpha=0.7, color="skyblue")
axes[0].set_title(f"Distribution of {DIST_COLUMN}")
axes[0].set_xlabel(DIST_COLUMN)
axes[0].set_ylabel("Frequency")
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(df[DIST_COLUMN], vert=True)
axes[1].set_title(f"Boxplot of {DIST_COLUMN}")
axes[1].set_ylabel(DIST_COLUMN)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.4 Custom relationship explorer

Like the **Relationship explorer** in the frontend, set `X_COL` and `Y_COL`
below and re-run the cell to see a scatter plot with a trend line.

In [ ]:
# Example pairs mirroring the frontend options:
# - 'phone_usage_hours' vs 'productivity_score'
# - 'study_hours_per_day' vs 'productivity_score'
# - 'sleep_hours' vs 'productivity_score'
# - 'focus_score' vs 'productivity_score'
# - 'stress_level' vs 'productivity_score'

X_COL = "phone_usage_hours"
Y_COL = "productivity_score"

if X_COL not in df.columns or Y_COL not in df.columns:
    raise ValueError(f"One or both columns not found: {X_COL!r}, {Y_COL!r}")

plt.figure(figsize=(7, 5))
plt.scatter(df[X_COL], df[Y_COL], alpha=0.4, s=20)
plt.xlabel(X_COL)
plt.ylabel(Y_COL)
plt.title(f"{X_COL} vs {Y_COL}")
plt.grid(True, alpha=0.3)

# Add a simple linear trend line
z = np.polyfit(df[X_COL], df[Y_COL], 1)
p = np.poly1d(z)
xs = np.linspace(df[X_COL].min(), df[X_COL].max(), 200)
plt.plot(xs, p(xs), "r--", linewidth=2)

plt.tight_layout()
plt.show()

In [ ]:
# 5. Optional: quick visual insights (similar to the dashboard)
sns.set_style("whitegrid")

# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()
